# Notebook 1: Transformer Fundamentals From Scratch

**Goal:** Implement every core component of the Transformer architecture from scratch using only NumPy (then PyTorch). This is exactly what Mistral's live coding round tests.

**Topics:**
1. Softmax (numerically stable)
2. Scaled Dot-Product Attention
3. Multi-Head Attention
4. Positional Encoding (Sinusoidal + RoPE)
5. Layer Normalization
6. Feed-Forward Network (with SwiGLU)
7. Full Transformer Block
8. Causal Mask
9. Mini Autoregressive Inference Loop

**Rules:** Implement each function yourself before looking at the reference. Each section has a `# YOUR CODE HERE` block.

In [ ]:
import numpy as np
import math
from typing import Optional, Tuple

np.random.seed(42)
print('NumPy version:', np.__version__)

---
## 1. Softmax — Numerically Stable

The naive `exp(x) / sum(exp(x))` overflows for large values.

**Fix:** subtract `max(x)` before exponentiating — doesn't change output mathematically.

**Formula:** `softmax(x_i) = exp(x_i - max(x)) / sum(exp(x_j - max(x)))`

**Interview question:** *Why is numerical stability important here? What error does naive softmax cause?*

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax along a given axis.
    
    Args:
        x: Input array of any shape
        axis: Axis along which to compute softmax
    Returns:
        Array of same shape, values sum to 1 along axis
    """
    # YOUR CODE HERE
    # Step 1: subtract max for numerical stability
    # Step 2: exponentiate
    # Step 3: divide by sum
    raise NotImplementedError()


# --- TESTS ---
def test_softmax():
    # Basic correctness: outputs sum to 1
    x = np.array([1.0, 2.0, 3.0, 4.0])
    out = softmax(x)
    assert np.isclose(out.sum(), 1.0), f'Sum should be 1, got {out.sum()}'
    assert out.shape == x.shape
    
    # Numerical stability: should not produce NaN/Inf
    x_large = np.array([1000.0, 1001.0, 1002.0])
    out_large = softmax(x_large)
    assert not np.any(np.isnan(out_large)), 'Should not produce NaN'
    assert not np.any(np.isinf(out_large)), 'Should not produce Inf'
    
    # 2D case
    x_2d = np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
    out_2d = softmax(x_2d, axis=-1)
    assert np.allclose(out_2d.sum(axis=-1), [1.0, 1.0])
    
    print('All softmax tests passed!')

test_softmax()

In [ ]:
# REFERENCE IMPLEMENTATION (reveal after you try)
def softmax_ref(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x_max = np.max(x, axis=axis, keepdims=True)
    x_shifted = x - x_max
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# Monkey-patch so rest of notebook works
softmax = softmax_ref
test_softmax()

---
## 2. Scaled Dot-Product Attention

**Formula:** `Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V`

**Why scale by sqrt(d_k)?**  
Without scaling, the dot products grow large as d_k increases, pushing softmax into regions with very small gradients (vanishing gradients).

**Causal mask:** For autoregressive generation, future tokens must be masked (set to -inf before softmax).

In [ ]:
def scaled_dot_product_attention(
    Q: np.ndarray,  # (batch, seq, d_k)
    K: np.ndarray,  # (batch, seq, d_k)
    V: np.ndarray,  # (batch, seq, d_v)
    mask: Optional[np.ndarray] = None,  # (batch, 1, seq, seq) boolean mask; True = mask out
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute scaled dot-product attention.
    
    Returns:
        output: (batch, seq, d_v) weighted value vectors
        weights: (batch, seq, seq) attention weights for visualization
    """
    d_k = Q.shape[-1]
    
    # YOUR CODE HERE
    # Step 1: Compute attention scores QK^T / sqrt(d_k)
    # scores shape: (batch, seq_q, seq_k)
    
    # Step 2: Apply mask (set masked positions to -1e9)
    
    # Step 3: Softmax to get weights
    
    # Step 4: Weighted sum of values
    
    raise NotImplementedError()


# --- TESTS ---
def test_attention():
    batch, seq, d_k, d_v = 2, 4, 8, 8
    Q = np.random.randn(batch, seq, d_k)
    K = np.random.randn(batch, seq, d_k)
    V = np.random.randn(batch, seq, d_v)
    
    out, weights = scaled_dot_product_attention(Q, K, V)
    
    assert out.shape == (batch, seq, d_v), f'Wrong output shape: {out.shape}'
    assert weights.shape == (batch, seq, seq), f'Wrong weights shape: {weights.shape}'
    assert np.allclose(weights.sum(axis=-1), 1.0), 'Weights must sum to 1'
    print('All attention tests passed!')

test_attention()

In [ ]:
# REFERENCE
def scaled_dot_product_attention_ref(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    d_k = Q.shape[-1]
    
    # (batch, seq_q, seq_k)
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    
    weights = softmax(scores, axis=-1)
    output = np.matmul(weights, V)
    return output, weights

scaled_dot_product_attention = scaled_dot_product_attention_ref
test_attention()

---
## 3. Causal Mask

For autoregressive models (GPT-style), token at position i can only attend to positions 0..i.
We mask future positions by setting their scores to -inf before softmax.

**Interview question:** *Why do we use -inf (or -1e9) rather than 0?*  
Because 0 after softmax becomes a small positive value. -inf becomes exactly 0 after softmax, which means no attention to future tokens.

In [ ]:
def make_causal_mask(seq_len: int) -> np.ndarray:
    """
    Create a causal (upper-triangular) mask.
    True means 'mask out this position' (future token).
    
    Returns shape: (seq_len, seq_len)
    Example for seq_len=4:
        [[F, T, T, T],
         [F, F, T, T],
         [F, F, F, T],
         [F, F, F, F]]
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# Test it visually
mask = make_causal_mask(4)
print('Causal mask (True = masked):')
print(mask.astype(int))

In [ ]:
# REFERENCE
def make_causal_mask_ref(seq_len: int) -> np.ndarray:
    # Upper triangular excluding diagonal = future positions
    return np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)

make_causal_mask = make_causal_mask_ref
print(make_causal_mask(4).astype(int))

# Now test causal attention: position 0 should only attend to itself
batch, seq, d_k = 1, 4, 8
Q = K = V = np.random.randn(batch, seq, d_k)
mask = make_causal_mask(seq)[np.newaxis, :, :]
out, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
print('\nAttention weights (causal):')
print(weights[0].round(3))
print('Upper triangle should be ~0')

---
## 4. Multi-Head Attention

**Idea:** Run h attention heads in parallel on lower-dimensional projections, then concatenate.

**Why multiple heads?** Each head can attend to different representation subspaces — syntax, semantics, coreference etc.

**Shape flow:**
- Input: `(batch, seq, d_model)`
- Project: Q, K, V each → `(batch, seq, h, d_k)` where `d_k = d_model / h`
- Attend: each head → `(batch, h, seq, d_k)`
- Concat + project back: `(batch, seq, d_model)`

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Weight matrices (using He init for demonstration)
        scale = np.sqrt(2.0 / d_model)
        self.W_q = np.random.randn(d_model, d_model) * scale  # (d_model, d_model)
        self.W_k = np.random.randn(d_model, d_model) * scale
        self.W_v = np.random.randn(d_model, d_model) * scale
        self.W_o = np.random.randn(d_model, d_model) * scale  # output projection
    
    def split_heads(self, x: np.ndarray) -> np.ndarray:
        """
        Reshape x from (batch, seq, d_model) to (batch, num_heads, seq, d_k)
        """
        # YOUR CODE HERE
        # Hint: reshape then transpose
        raise NotImplementedError()
    
    def merge_heads(self, x: np.ndarray) -> np.ndarray:
        """
        Reverse split_heads: (batch, num_heads, seq, d_k) -> (batch, seq, d_model)
        """
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def forward(self, x: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
        """
        Args:
            x: (batch, seq, d_model)
            mask: optional causal mask
        Returns:
            (batch, seq, d_model)
        """
        # YOUR CODE HERE
        # Step 1: Project Q, K, V
        # Step 2: Split into heads
        # Step 3: Attention on each head
        # Step 4: Merge heads
        # Step 5: Output projection
        raise NotImplementedError()


# Test
batch, seq, d_model, num_heads = 2, 6, 64, 8
mha = MultiHeadAttention(d_model, num_heads)
x = np.random.randn(batch, seq, d_model)
# This will error until you implement it:
# out = mha.forward(x)
# print('MHA output shape:', out.shape)  # should be (2, 6, 64)

In [ ]:
# REFERENCE
class MultiHeadAttentionRef:
    def __init__(self, d_model: int, num_heads: int):
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        scale = np.sqrt(2.0 / d_model)
        self.W_q = np.random.randn(d_model, d_model) * scale
        self.W_k = np.random.randn(d_model, d_model) * scale
        self.W_v = np.random.randn(d_model, d_model) * scale
        self.W_o = np.random.randn(d_model, d_model) * scale
    
    def split_heads(self, x: np.ndarray) -> np.ndarray:
        batch, seq, _ = x.shape
        x = x.reshape(batch, seq, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)  # (batch, heads, seq, d_k)
    
    def merge_heads(self, x: np.ndarray) -> np.ndarray:
        batch, heads, seq, d_k = x.shape
        x = x.transpose(0, 2, 1, 3)  # (batch, seq, heads, d_k)
        return x.reshape(batch, seq, self.d_model)
    
    def forward(self, x: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
        batch, seq, _ = x.shape
        
        Q = x @ self.W_q  # (batch, seq, d_model)
        K = x @ self.W_k
        V = x @ self.W_v
        
        Q = self.split_heads(Q)  # (batch, heads, seq, d_k)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Reshape for batched attention across heads
        # Merge batch and heads dimensions
        Q_flat = Q.reshape(batch * self.num_heads, seq, self.d_k)
        K_flat = K.reshape(batch * self.num_heads, seq, self.d_k)
        V_flat = V.reshape(batch * self.num_heads, seq, self.d_k)
        
        attn_out, _ = scaled_dot_product_attention(Q_flat, K_flat, V_flat, mask=mask)
        attn_out = attn_out.reshape(batch, self.num_heads, seq, self.d_k)
        
        merged = self.merge_heads(attn_out)  # (batch, seq, d_model)
        return merged @ self.W_o

MultiHeadAttention = MultiHeadAttentionRef
batch, seq, d_model, num_heads = 2, 6, 64, 8
np.random.seed(42)
mha = MultiHeadAttention(d_model, num_heads)
x = np.random.randn(batch, seq, d_model)
out = mha.forward(x)
print('MHA output shape:', out.shape)  # (2, 6, 64)

---
## 5. Layer Normalization

**LayerNorm** normalizes across the feature dimension (not batch), making it batch-size independent.

**Formula:** `LN(x) = (x - mean(x)) / sqrt(var(x) + eps) * gamma + beta`

**Pre-LN vs Post-LN:**
- Post-LN (original Transformer): LayerNorm after residual → unstable at scale
- Pre-LN (GPT-2, Mistral): LayerNorm before sublayer → more stable training

In [ ]:
class LayerNorm:
    def __init__(self, d_model: int, eps: float = 1e-6):
        self.eps = eps
        self.gamma = np.ones(d_model)   # learnable scale
        self.beta = np.zeros(d_model)   # learnable shift
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        x: (..., d_model) — normalize over last dimension
        """
        # YOUR CODE HERE
        raise NotImplementedError()

# Test
ln = LayerNorm(64)
x = np.random.randn(2, 6, 64) * 10 + 5
# out = ln.forward(x)
# print('Mean ~0:', out[0,0].mean().round(4))
# print('Std ~1:', out[0,0].std().round(4))

In [ ]:
# REFERENCE
class LayerNormRef:
    def __init__(self, d_model: int, eps: float = 1e-6):
        self.eps = eps
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

LayerNorm = LayerNormRef
ln = LayerNorm(64)
x = np.random.randn(2, 6, 64) * 10 + 5
out = ln.forward(x)
print('Mean ~0:', out[0,0].mean().round(4))
print('Std ~1:', out[0,0].std().round(4))

---
## 6. Feed-Forward Network (FFN) with SwiGLU

Standard FFN: `FFN(x) = relu(xW_1 + b_1)W_2 + b_2` — expand then contract

**SwiGLU** (used in Llama/Mistral): `FFN(x) = (swish(xW_1) * xW_3) W_2`
- Swish: `x * sigmoid(x)`
- Uses 3 matrices instead of 2 for the gating
- Better empirical performance than ReLU/GELU

In [ ]:
def swish(x: np.ndarray) -> np.ndarray:
    """Swish activation: x * sigmoid(x)"""
    # YOUR CODE HERE
    raise NotImplementedError()

class SwiGLU_FFN:
    def __init__(self, d_model: int, d_ff: int):
        scale = 0.02
        self.W1 = np.random.randn(d_model, d_ff) * scale  # gate
        self.W2 = np.random.randn(d_ff, d_model) * scale  # down
        self.W3 = np.random.randn(d_model, d_ff) * scale  # value
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        SwiGLU: swish(x @ W1) * (x @ W3) then project down
        x: (batch, seq, d_model)
        """
        # YOUR CODE HERE
        raise NotImplementedError()

# Test
ffn = SwiGLU_FFN(64, 256)
x = np.random.randn(2, 6, 64)
# out = ffn.forward(x)
# print('FFN output shape:', out.shape)  # (2, 6, 64)

In [ ]:
# REFERENCE
def swish_ref(x: np.ndarray) -> np.ndarray:
    return x / (1.0 + np.exp(-x))

class SwiGLU_FFN_Ref:
    def __init__(self, d_model: int, d_ff: int):
        scale = 0.02
        self.W1 = np.random.randn(d_model, d_ff) * scale
        self.W2 = np.random.randn(d_ff, d_model) * scale
        self.W3 = np.random.randn(d_model, d_ff) * scale
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        gate = swish_ref(x @ self.W1)  # (batch, seq, d_ff)
        val = x @ self.W3             # (batch, seq, d_ff)
        return (gate * val) @ self.W2  # (batch, seq, d_model)

swish = swish_ref
SwiGLU_FFN = SwiGLU_FFN_Ref
ffn = SwiGLU_FFN(64, 256)
x = np.random.randn(2, 6, 64)
out = ffn.forward(x)
print('FFN output shape:', out.shape)

---
## 7. Full Transformer Decoder Block (Pre-LN style)

**Architecture (GPT/Mistral style — Pre-LN):**
```
x -> LN -> MHA -> + (residual) -> LN -> FFN -> + (residual) -> output
```

In [ ]:
class TransformerBlock:
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        self.ln1 = LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = LayerNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff)
    
    def forward(self, x: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
        """
        Pre-LN Transformer block with residual connections.
        x: (batch, seq, d_model)
        """
        # YOUR CODE HERE
        # x = x + attn(ln1(x))
        # x = x + ffn(ln2(x))
        raise NotImplementedError()

# block = TransformerBlock(64, 8, 256)
# x = np.random.randn(2, 6, 64)
# out = block.forward(x)
# print('Block output shape:', out.shape)

In [ ]:
# REFERENCE
class TransformerBlockRef:
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        self.ln1 = LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = LayerNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff)
    
    def forward(self, x: np.ndarray, mask: Optional[np.ndarray] = None) -> np.ndarray:
        # Pre-LN: normalize BEFORE sublayer, then add residual
        x = x + self.attn.forward(self.ln1.forward(x), mask=mask)
        x = x + self.ffn.forward(self.ln2.forward(x))
        return x

TransformerBlock = TransformerBlockRef

np.random.seed(42)
block = TransformerBlock(64, 8, 256)
x = np.random.randn(2, 6, 64)
mask = make_causal_mask(6)[np.newaxis, :, :]
out = block.forward(x, mask=mask)
print('Block output shape:', out.shape)  # (2, 6, 64)

---
## 8. Sinusoidal Positional Encoding

Without positional encoding, the Transformer is permutation-invariant.

**Formula:**
- `PE(pos, 2i) = sin(pos / 10000^(2i/d_model))`
- `PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))`

In [ ]:
def sinusoidal_position_encoding(seq_len: int, d_model: int) -> np.ndarray:
    """
    Returns: (seq_len, d_model) positional encoding matrix
    """
    # YOUR CODE HERE
    # For each position pos and each dimension i:
    # PE[pos, 2i] = sin(pos / 10000^(2i/d_model))
    # PE[pos, 2i+1] = cos(pos / 10000^(2i/d_model))
    raise NotImplementedError()

# pe = sinusoidal_position_encoding(10, 64)
# print('PE shape:', pe.shape)

# Visualize
# import matplotlib.pyplot as plt
# plt.figure(figsize=(12, 4))
# plt.imshow(sinusoidal_position_encoding(50, 128).T, aspect='auto', cmap='RdBu')
# plt.xlabel('Position'); plt.ylabel('Dimension'); plt.colorbar()
# plt.title('Sinusoidal Positional Encoding')
# plt.show()

In [ ]:
# REFERENCE
def sinusoidal_position_encoding_ref(seq_len: int, d_model: int) -> np.ndarray:
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, np.newaxis]  # (seq_len, 1)
    div_term = np.power(10000.0, np.arange(0, d_model, 2) / d_model)  # (d_model/2,)
    
    pe[:, 0::2] = np.sin(position / div_term)  # even dims
    pe[:, 1::2] = np.cos(position / div_term)  # odd dims
    return pe

sinusoidal_position_encoding = sinusoidal_position_encoding_ref
pe = sinusoidal_position_encoding(50, 64)
print('PE shape:', pe.shape)
print('First few values:', pe[0, :8].round(3))

---
## 9. PyTorch Version — Production-Quality MHA

Now implement the same in PyTorch with proper typing and using `torch.nn.Module`.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    print('PyTorch version:', torch.__version__)
    TORCH_AVAILABLE = True
except ImportError:
    print('PyTorch not installed. Run: pip install torch')
    TORCH_AVAILABLE = False

In [ ]:
if TORCH_AVAILABLE:
    class TorchMultiHeadAttention(nn.Module):
        def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
            super().__init__()
            assert d_model % num_heads == 0
            self.num_heads = num_heads
            self.d_k = d_model // num_heads
            
            # YOUR CODE HERE: define projection layers
            # self.W_q = ...
            # self.W_k = ...
            # self.W_v = ...
            # self.W_o = ...
            # self.dropout = ...
            raise NotImplementedError()
        
        def forward(
            self,
            x: torch.Tensor,  # (B, T, d_model)
            mask: Optional[torch.Tensor] = None,
        ) -> torch.Tensor:
            # YOUR CODE HERE
            raise NotImplementedError()

    print('Implement TorchMultiHeadAttention above')

In [ ]:
# REFERENCE PyTorch MHA
if TORCH_AVAILABLE:
    class TorchMHARef(nn.Module):
        def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
            super().__init__()
            assert d_model % num_heads == 0
            self.num_heads = num_heads
            self.d_k = d_model // num_heads
            
            self.W_q = nn.Linear(d_model, d_model, bias=False)
            self.W_k = nn.Linear(d_model, d_model, bias=False)
            self.W_v = nn.Linear(d_model, d_model, bias=False)
            self.W_o = nn.Linear(d_model, d_model, bias=False)
            self.dropout = nn.Dropout(dropout)
        
        def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
            B, T, C = x.shape
            
            Q = self.W_q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
            K = self.W_k(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
            V = self.W_v(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
            # Q, K, V: (B, heads, T, d_k)
            
            scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, heads, T, T)
            
            if mask is not None:
                scores = scores.masked_fill(mask, float('-inf'))
            
            weights = F.softmax(scores, dim=-1)
            weights = self.dropout(weights)
            
            out = weights @ V  # (B, heads, T, d_k)
            out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
            return self.W_o(out)
    
    # Test
    torch.manual_seed(42)
    mha_torch = TorchMHARef(64, 8)
    x_t = torch.randn(2, 6, 64)
    causal = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1).unsqueeze(0).unsqueeze(0)
    out_t = mha_torch(x_t, mask=causal)
    print('PyTorch MHA output shape:', out_t.shape)  # (2, 6, 64)

---
## 10. Practice Questions (Answer These Verbally)

These are real questions from Mistral/similar LLM company interviews:

1. **Complexity:** What is the time complexity of self-attention with respect to sequence length? How does Sliding Window Attention (Mistral's approach) improve this?

2. **GQA:** What is Grouped Query Attention? If we have 32 query heads but only 8 KV heads, how much memory does this save?

3. **FlashAttention:** Why is naive attention memory-inefficient? How does FlashAttention fix it? (Hint: IO complexity, tiling)

4. **RoPE:** How does RoPE encode position differently from sinusoidal PE? Why does it generalize better to longer sequences?

5. **Debug scenario:** Your training loss spikes at step 40k after being stable. What do you check first?

6. **Architecture choice:** Why does Mistral use Pre-LN instead of Post-LN? What happens to gradients at scale?

---

**Next:** Go to `02_tokenization_and_sampling.ipynb`